In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Spark SQL").getOrCreate()

26/05/05 13:21:27 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [4]:
listings = spark.read.csv("listings.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")

In [5]:
reviews = spark.read.csv("reviews.csv.gz", header=True, inferSchema=True, sep=",", quote='"', escape='"', multiLine=True, mode="PERMISSIVE")

In [7]:
listing_reviews = listings.join(reviews, listings.id == reviews.listing_id, how='inner')

In [8]:
reviews_per_listing = listing_reviews.groupBy(listings.id, listings.name).agg(F.count(reviews.id).alias('num_reviews')).orderBy('num_reviews', ascending=False).show(truncate=False)

[Stage 5:>                                                          (0 + 1) / 1]

+--------+--------------------------------------------------+-----------+
|id      |name                                              |num_reviews|
+--------+--------------------------------------------------+-----------+
|47408549|Double Room+ Ensuite                              |1902       |
|43120947|Private double room with en suite facilities      |1647       |
|19670926|Locke Studio Apartment at Leman Locke             |1443       |
|2126708 |London's best transport hub 5 mins walk! Safe too!|1142       |
|46233904|Superior Studio, avg size 23.5 msq                |1002       |
|2659707 |Large Room + Private Bathroom, E3.                |998        |
|27833488|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951        |
|4748665 |Single bedroom near London Stratford              |933        |
|42081759|Micro Studio at Locke at Broken Wharf             |914        |
|5266466 |Large London Room, Ensuite Bathroom,TV & Breakfast|909        |
|2025844 |Family Friendly Central Lond

In [9]:
listings.createOrReplaceTempView("listings")
reviews.createOrReplaceTempView("reviews")

26/05/05 13:25:51 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [10]:
query = """select listings.id, listings.name, count(reviews.id) as num_reviews from listings
inner join reviews on listings.id = reviews.listing_id
group by listings.id, listings.name
order by num_reviews desc"""

reviews_per_listing = spark.sql(query)
reviews_per_listing.show(truncate=False)

[Stage 9:>                                                          (0 + 1) / 1]

+--------+--------------------------------------------------+-----------+
|id      |name                                              |num_reviews|
+--------+--------------------------------------------------+-----------+
|47408549|Double Room+ Ensuite                              |1902       |
|43120947|Private double room with en suite facilities      |1647       |
|19670926|Locke Studio Apartment at Leman Locke             |1443       |
|2126708 |London's best transport hub 5 mins walk! Safe too!|1142       |
|46233904|Superior Studio, avg size 23.5 msq                |1002       |
|2659707 |Large Room + Private Bathroom, E3.                |998        |
|27833488|S - Heathrow Airport Terminal 2 3 4 5 Hatton Cross|951        |
|4748665 |Single bedroom near London Stratford              |933        |
|42081759|Micro Studio at Locke at Broken Wharf             |914        |
|5266466 |Large London Room, Ensuite Bathroom,TV & Breakfast|909        |
|2025844 |Family Friendly Central Lond